# Interpretación y análisis de fechas

## Breve repaso

En el trabajo anterior sobre conversión de tipos de datos vimos que no alcanza con que una columna parezca numérica o parezca una fecha. Para que Pandas pueda trabajar correctamente con esos valores, necesita interpretarlos con el tipo adecuado.

Convertimos columnas como `Quantity`, `Price Per Unit` y `Total Spent` a formato numérico usando `pd.to_numeric()`. También hicimos una primera conversión de la columna `Transaction Date` usando `pd.to_datetime()`.

En esta práctica trabajaremos con profundizar en el trabajo con fechas.

Las fechas son especialmente relevantes porque permiten analizar los datos a lo largo del tiempo. En un dataset de ventas, por ejemplo, una columna de fecha puede ayudarnos a responder preguntas como: ¿en qué días hubo más transacciones?, ¿qué período cubre el dataset?, ¿hay ventas concentradas en ciertos días?, ¿cómo se distribuyen las operaciones a lo largo del tiempo?

Para responder este tipo de preguntas, no alcanza con que la fecha esté escrita como texto. Necesitamos que Pandas la interprete como dato temporal.

Una vez que una columna fue convertida a fecha, es posible ordenar registros cronológicamente, detectar valores fuera de rango, extraer el año, el mes o el día, y construir nuevas variables temporales útiles para el análisis.

También trabajaremos con hacer un primer uso acotado de `groupby()`. Esta es una herramienta muy potente de Pandas para agrupar datos y calcular resúmenes por grupo. Más adelante la trabajaremos con estudiar con mayor profundidad. En este sección la usaremos solamente para mostrar cómo una fecha bien convertida permite resumir información por períodos.

Al finalizar este notebook deberías poder:

- Comprender por qué las fechas necesitan un tratamiento especial.
- Convertir una columna de texto a fecha usando `pd.to_datetime()`.
- Identificar valores temporales no interpretables.
- Revisar el rango temporal de un dataset.
- Ordenar datos usando una columna de fecha.
- Extraer componentes como año, mes, día y día de la semana.
- Crear columnas nuevas a partir de una fecha.
- Realizar primeras exploraciones temporales simples.

### Punto de control

Una fecha almacenada como texto puede parecer correcta en una muestra y aun así impedir operaciones temporales. El tipo de dato es el que habilita ordenar, comparar y agrupar cronológicamente.

## Archivo de trabajo

En este sección analizaremos fechas utilizando el dataset **Cafe Sales — Dirty Data for Cleaning Training**. El archivo debe ser cargado por el usuario directamente en Google Colab mediante el selector que aparece en la celda de carga.

La variable central será `Transaction Date`, que contiene la fecha asociada a cada transacción. Después de seleccionar el CSV, comenzaremos inspeccionando cómo Pandas interpretó esa columna y revisaremos algunos valores de ejemplo.

El recorrido será progresivo: primero confirmaremos que el archivo fue recibido, luego convertiremos la columna a un tipo temporal y finalmente comprobaremos los valores que no pudieron interpretarse.

Este notebook no descarga datos desde internet ni depende de una ruta local. Para ejecutarlo, el usuario debe subir un archivo `.csv` antes de continuar con las celdas de análisis.

### Punto de control

Trabajar con una copia del `DataFrame` permite conservar una referencia de la fuente original y comparar los efectos de la conversión.

In [ ]:
# El usuario debe seleccionar el CSV desde su equipo en Google Colab.
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()
archivos = list(uploaded.keys())
archivos_csv = [archivo for archivo in archivos if archivo.lower().endswith('.csv')]

if not archivos_csv:
    raise ValueError('Debes subir al menos un archivo con extensión .csv')

nombre_csv = archivos_csv[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre_csv]))

columnas_requeridas = {
    'Transaction ID', 'Item', 'Quantity', 'Price Per Unit',
    'Total Spent', 'Payment Method', 'Location', 'Transaction Date'
}
columnas_faltantes = columnas_requeridas - set(df.columns)
if columnas_faltantes:
    raise ValueError(
        'El archivo no corresponde al dataset esperado. '
        f'Faltan columnas: {sorted(columnas_faltantes)}'
    )

df.head()

La salida de `head()` nos permite confirmar que el archivo fue cargado correctamente.

En este sección trabajaremos con concentrarnos en la columna `Transaction Date`. Antes de transformarla, es necesario revisar cómo fue cargada por Pandas.

### Punto de control

Usar `errors="coerce"` evita que una sola cadena defectuosa detenga todo el proceso, pero convierte esos casos en `NaT`. Por eso siempre debe acompañarse de un conteo y una revisión.

## Diagnóstico de la variable temporal

Antes de convertir una columna, resulta útil observar cómo fue cargada.

En este dataset, la columna que registra la fecha de cada transacción se llama `Transaction Date`.

Podemos revisar su tipo de dato:

### Punto de control

Las fechas no interpretables no deben desaparecer automáticamente. Pueden conservar información útil en otras columnas y deberán tratarse según el objetivo del análisis.

In [ ]:
df["Transaction Date"].dtype

La salida indica que la columna fue cargada como `object`.

Eso significa que, por ahora, Pandas la está tratando como texto. Aunque los valores parezcan fechas, todavía no son fechas en sentido técnico para Pandas.

Podemos observar algunos valores:

### Punto de control

El rango temporal permite detectar periodos inesperados, fechas fuera de contexto o concentraciones que pueden influir en la lectura de las ventas.

In [ ]:
df["Transaction Date"].head(10)

Visualmente, los valores pueden parecer fechas válidas. Pero mientras la columna esté como `object`, no podremos aprovechar correctamente las herramientas temporales de Pandas.

También es posible revisar cuántos valores faltantes tiene esta columna antes de convertirla:

### Punto de control

Las columnas derivadas no sustituyen a la fecha original. Funcionan como variables auxiliares para responder preguntas específicas sobre años, meses, días y patrones de actividad.

In [ ]:
df["Transaction Date"].isna().sum()

Este conteo nos da un primer diagnóstico.

Más adelante trabajaremos con convertir la columna con `pd.to_datetime()` y compararemos si la cantidad de valores faltantes cambia. Si aumenta, eso indicará que había valores escritos como texto que no pudieron interpretarse como fechas.

### Punto de control

Cuando se agrupan transacciones, el nivel de detalle cambia: se pasa de registros individuales a resúmenes por fecha o periodo. Esa transformación debe interpretarse con claridad.

## Transformar texto en fechas

Ya vimos que `Transaction Date` fue cargada como `object`. Aunque sus valores parecen fechas, Pandas todavía la trata como texto.

Para convertirla a un tipo temporal usamos `pd.to_datetime()`.

Vamos a crear una copia del dataset para trabajar sin modificar directamente el `DataFrame` original.

### Punto de control

Una validación temporal debe revisar tanto el tipo de columna como los valores extremos, los nulos y la coherencia del orden cronológico.

In [ ]:
df_fechas = df.copy()

df_fechas["Transaction Date"] = pd.to_datetime(
    df_fechas["Transaction Date"],
    errors="coerce"
)

df_fechas["Transaction Date"].head(10)

A continuación la columna fue convertida a formato de fecha.

Usamos `errors="coerce"` para que cualquier valor que no pueda interpretarse como fecha se convierta en `NaT`.

`NaT` significa *Not a Time*. Es el equivalente temporal de un valor faltante. Así como `NaN` representa un dato faltante en muchas columnas, `NaT` representa una fecha faltante o no interpretable.

Después de convertir, debemos verificar el tipo de dato resultante.

### Punto de control

Una fecha almacenada como texto puede parecer correcta en una muestra y aun así impedir operaciones temporales. El tipo de dato es el que habilita ordenar, comparar y agrupar cronológicamente.

In [ ]:
df_fechas["Transaction Date"].dtype

El tipo `datetime64[ns]` indica que Pandas ya reconoce la columna como temporal.

Esto nos permite trabajar con herramientas específicas para fechas: ordenar cronológicamente, extraer año, mes o día, calcular rangos temporales y construir nuevas columnas derivadas de la fecha.

### Punto de control

La conversión se realiza sobre una copia para conservar una referencia del estado original. Esta separación facilita revisar qué valores cambiaron y qué casos requieren atención adicional.

## Detectar conversiones fallidas

Antes de convertir la columna `Transaction Date`, ya habíamos detectado 159 valores faltantes.

Después de usar `pd.to_datetime()` con `errors="coerce"`, resulta útil revisar si esa cantidad cambió.

Si la cantidad de faltantes aumenta, significa que algunos valores que no estaban vacíos no pudieron interpretarse como fechas y fueron convertidos en `NaT`.

### Punto de control

Usar `errors="coerce"` evita que una sola cadena defectuosa detenga todo el proceso, pero convierte esos casos en `NaT`. Por eso siempre debe acompañarse de un conteo y una revisión.

In [ ]:
faltantes_fecha_antes = df["Transaction Date"].isna().sum()
faltantes_fecha_despues = df_fechas["Transaction Date"].isna().sum()

print("Faltantes antes de convertir:")
print(faltantes_fecha_antes)

print()

print("Faltantes después de convertir:")
print(faltantes_fecha_despues)

Si ambos valores coinciden, entonces la conversión no generó nuevos faltantes. Eso significa que los valores no faltantes de la columna pudieron interpretarse correctamente como fechas.

Si el segundo valor fuera mayor que el primero, deberíamos investigar qué valores no pudieron convertirse.

Esta comparación es relevante porque `errors="coerce"` evita que la conversión se detenga con un error, pero también puede transformar valores problemáticos en `NaT`. Por esa razón, después de convertir fechas, siempre debemos verificar cuántos valores temporales faltantes quedaron.

### Punto de control

Las fechas no interpretables no deben desaparecer automáticamente. Pueden conservar información útil en otras columnas y deberán tratarse según el objetivo del análisis.

El resultado muestra algo relevante: después de convertir, la cantidad de valores faltantes aumentó.

Antes de la conversión había 159 valores faltantes en `Transaction Date`. Después de aplicar `pd.to_datetime()` con `errors="coerce"`, aparecen 460 valores temporales faltantes.

Eso significa que había valores no vacíos que Pandas no pudo interpretar como fechas. Al usar `errors="coerce"`, esos valores fueron convertidos en `NaT`.

Calculemos cuántos valores nuevos se volvieron faltantes durante la conversión.

### Punto de control

El rango temporal permite detectar periodos inesperados, fechas fuera de contexto o concentraciones que pueden influir en la lectura de las ventas.

In [ ]:
nuevos_nat = faltantes_fecha_despues - faltantes_fecha_antes

nuevos_nat

El resultado indica cuántos valores no faltantes se convirtieron en `NaT`.

Estos casos merecen una revisión adicional. No eran valores vacíos, pero tampoco pudieron convertirse correctamente a fecha. Podrían ser textos como `"ERROR"`, `"UNKNOWN"` u otros valores problemáticos.

El siguiente paso es identificar cuáles eran esos valores originales.

### Punto de control

Las columnas derivadas no sustituyen a la fecha original. Funcionan como variables auxiliares para responder preguntas específicas sobre años, meses, días y patrones de actividad.

## Investigar fechas problemáticas

A continuación sabemos que la conversión generó nuevos valores `NaT`.

Para investigar esos casos, es necesario comparar dos cosas:

```text
la columna original antes de convertir
la columna convertida a fecha
```

Nos interesan las filas donde la columna original no estaba vacía, pero la columna convertida quedó como `NaT`.

Podemos construir una condición para identificar esos casos.

### Punto de control

Cuando se agrupan transacciones, el nivel de detalle cambia: se pasa de registros individuales a resúmenes por fecha o periodo. Esa transformación debe interpretarse con claridad.

In [ ]:
fechas_no_convertidas = (
    df["Transaction Date"].notna()
    & df_fechas["Transaction Date"].isna()
)

fechas_no_convertidas.sum()

El resultado debería coincidir con la cantidad de nuevos valores `NaT`.

A continuación es posible observar qué valores originales tenían esas filas.

### Punto de control

Una validación temporal debe revisar tanto el tipo de columna como los valores extremos, los nulos y la coherencia del orden cronológico.

In [ ]:
df.loc[fechas_no_convertidas, "Transaction Date"].value_counts(dropna=False)

Este resultado muestra cuáles eran los valores originales que no pudieron interpretarse como fechas.

Si aparecen valores como `"UNKNOWN"` o `"ERROR"`, eso confirma que la columna no tenía solamente fechas válidas y valores faltantes, sino también textos problemáticos.

Esta revisión es relevante porque nos ayuda a diferenciar situaciones:

```text
NaN original        → la fecha ya estaba faltante antes de convertir
NaT por conversión  → había un valor escrito, pero no era una fecha interpretable
```

Ambos casos aparecen como faltantes después de la conversión, pero no significan exactamente lo mismo. Esa diferencia puede ser relevante si queremos documentar la calidad del dataset.

### Punto de control

Una fecha almacenada como texto puede parecer correcta en una muestra y aun así impedir operaciones temporales. El tipo de dato es el que habilita ordenar, comparar y agrupar cronológicamente.

## Delimitar el periodo de los datos

Una vez que la columna fue convertida a fecha, es posible empezar a obtener información temporal útil.

Una primera pregunta relevante es:

```text
¿Qué período cubre el dataset?
```

Para responderla es posible buscar la fecha mínima y la fecha máxima registradas.

### Punto de control

La conversión se realiza sobre una copia para conservar una referencia del estado original. Esta separación facilita revisar qué valores cambiaron y qué casos requieren atención adicional.

In [ ]:
fecha_minima = df_fechas["Transaction Date"].min()
fecha_maxima = df_fechas["Transaction Date"].max()

print("Fecha mínima:")
print(fecha_minima)

print()

print("Fecha máxima:")
print(fecha_maxima)

La fecha mínima indica el primer momento registrado en el dataset. La fecha máxima indica el último.

Esta revisión nos permite entender el rango temporal de los datos. Por ejemplo, podríamos descubrir que el dataset cubre varios meses, un año completo o solo un período breve.

También puede servir para detectar valores sospechosos. Si esperamos trabajar con ventas de un año determinado y aparece una fecha demasiado antigua o demasiado futura, esa fecha debería revisarse.

Cuando una columna está cargada como texto, este tipo de revisión puede ser menos confiable. Al convertirla a fecha, Pandas puede comparar los valores temporalmente y encontrar correctamente el mínimo y el máximo.

### Punto de control

Usar `errors="coerce"` evita que una sola cadena defectuosa detenga todo el proceso, pero convierte esos casos en `NaT`. Por eso siempre debe acompañarse de un conteo y una revisión.

También es posible calcular la amplitud temporal del dataset, es decir, cuánto tiempo transcurre entre la primera y la última fecha registrada.

### Punto de control

Las fechas no interpretables no deben desaparecer automáticamente. Pueden conservar información útil en otras columnas y deberán tratarse según el objetivo del análisis.

In [ ]:
fecha_maxima - fecha_minima

El resultado es una diferencia de tiempo.

Este valor nos da una idea general de la extensión temporal del dataset. A partir de esta información, más adelante podríamos decidir si tiene sentido analizar los datos por día, por mes, por trimestre o por otro período.

### Punto de control

El rango temporal permite detectar periodos inesperados, fechas fuera de contexto o concentraciones que pueden influir en la lectura de las ventas.

## Organizar los registros cronológicamente

Una vez que una columna fue convertida a tipo temporal, es posible usarla para ordenar el dataset cronológicamente.

Ordenar por fecha permite leer las transacciones en el orden en que ocurrieron. Esto puede ser útil para revisar la evolución de los datos, detectar períodos con mayor actividad o preparar análisis posteriores.

Vamos a ordenar el `DataFrame` según `Transaction Date`.

### Punto de control

Las columnas derivadas no sustituyen a la fecha original. Funcionan como variables auxiliares para responder preguntas específicas sobre años, meses, días y patrones de actividad.

In [ ]:
df_fechas.sort_values("Transaction Date").head(10)

El resultado permite observar las primeras transacciones según el orden temporal.

Como la columna ya fue convertida a fecha, Pandas no ordena los valores como simples textos, sino como fechas reales. Esto es relevante porque el orden cronológico depende del significado temporal de los valores, no solamente de cómo están escritos.

También es posible mirar las últimas fechas del dataset:

### Punto de control

Cuando se agrupan transacciones, el nivel de detalle cambia: se pasa de registros individuales a resúmenes por fecha o periodo. Esa transformación debe interpretarse con claridad.

In [ ]:
df_fechas.sort_values("Transaction Date").tail(10)

Esta vista muestra las transacciones ubicadas al final del período registrado.

Al ordenar por fecha, es posible que los valores `NaT` aparezcan al final del resultado. Esto ocurre porque esos registros no tienen una fecha válida para ser ubicados cronológicamente.

Por esa razón, cuando trabajamos con fechas, debemos recordar que no todos los registros necesariamente podrán participar de un análisis temporal. Las filas con fechas faltantes o no interpretables requieren una decisión específica: conservarlas para otros análisis, excluirlas de ciertos cálculos temporales o revisarlas con más detalle.

### Punto de control

Una validación temporal debe revisar tanto el tipo de columna como los valores extremos, los nulos y la coherencia del orden cronológico.

## Derivar partes de la fecha

Una vez que una columna está convertida a tipo fecha, es posible extraer partes específicas de cada valor temporal.

Por ejemplo, de una fecha es posible obtener:

```text
año
mes
día
día de la semana
```

Esto es útil porque muchas preguntas de análisis no se hacen sobre la fecha completa, sino sobre alguna de sus partes. Por ejemplo, podríamos querer saber si hay más transacciones en ciertos meses, si algunos días de la semana tienen más actividad o si el dataset cubre un solo año o varios.

En Pandas, para acceder a componentes de una fecha usamos `.dt`.

Primero trabajaremos con crear una copia de trabajo con la columna de fecha ya convertida.

### Punto de control

Una fecha almacenada como texto puede parecer correcta en una muestra y aun así impedir operaciones temporales. El tipo de dato es el que habilita ordenar, comparar y agrupar cronológicamente.

In [ ]:
df_temporal = df_fechas.copy()

df_temporal["anio"] = df_temporal["Transaction Date"].dt.year
df_temporal["mes"] = df_temporal["Transaction Date"].dt.month
df_temporal["dia"] = df_temporal["Transaction Date"].dt.day

df_temporal[
    [
        "Transaction Date",
        "anio",
        "mes",
        "dia"
    ]
].head(10)

A continuación el `DataFrame` tiene nuevas columnas derivadas de la fecha.

La columna `anio` contiene el año de la transacción. La columna `mes` contiene el número de mes. La columna `dia` contiene el día del mes.

Observemos que los valores pueden aparecer con `.0`, por ejemplo `2023.0` o `9.0`. Esto ocurre porque algunas filas tienen `NaT` en la fecha original. Al extraer año, mes o día de una fecha faltante, Pandas genera valores faltantes en las columnas derivadas, y por eso usa un formato numérico que puede representar esos faltantes.

Lo relevante en este punto no es el formato visual, sino la idea: las columnas `anio`, `mes` y `dia` fueron construidas a partir de la fecha convertida.

Estas columnas no estaban en el dataset original. Las construimos a partir de `Transaction Date`.

Este tipo de transformación es muy útil porque permite preparar el dataset para preguntas temporales más específicas. En lugar de trabajar siempre con la fecha completa, es posible analizar los datos por año, mes o día.

### Punto de control

La conversión se realiza sobre una copia para conservar una referencia del estado original. Esta separación facilita revisar qué valores cambiaron y qué casos requieren atención adicional.

## Incorporar el día de la semana

Además del año, el mes y el día del mes, también es posible extraer el día de la semana.

Esto puede ser útil para analizar patrones de comportamiento. Por ejemplo, en un comercio podríamos preguntarnos si hay más transacciones los fines de semana, si ciertos productos se venden más en determinados días o si la actividad cambia entre días laborales y no laborales.

Pandas permite obtener el nombre del día usando `.dt.day_name()`.

### Punto de control

Usar `errors="coerce"` evita que una sola cadena defectuosa detenga todo el proceso, pero convierte esos casos en `NaT`. Por eso siempre debe acompañarse de un conteo y una revisión.

In [ ]:
df_temporal["dia_semana"] = df_temporal["Transaction Date"].dt.day_name()

df_temporal[
    [
        "Transaction Date",
        "dia_semana"
    ]
].head(10)

La columna `dia_semana` contiene el nombre del día correspondiente a cada fecha.

Por defecto, en este entorno los nombres de los días aparecen en inglés, por ejemplo `Friday`, `Tuesday` o `Wednesday`. Esto no afecta el análisis: siguen representando correctamente el día de la semana. Más adelante, si necesitáramos presentar los resultados en español, podríamos crear un reemplazo o mapeo de nombres.

Esta información no estaba escrita directamente en el dataset original. La obtuvimos a partir de la columna temporal.

### Punto de control

Las fechas no interpretables no deben desaparecer automáticamente. Pueden conservar información útil en otras columnas y deberán tratarse según el objetivo del análisis.

In [ ]:
df_temporal["numero_dia_semana"] = df_temporal["Transaction Date"].dt.dayofweek

df_temporal[
    [
        "Transaction Date",
        "dia_semana",
        "numero_dia_semana"
    ]
].head(10)

En Pandas, `dayofweek` usa la siguiente convención:

```text
0 → lunes
1 → martes
2 → miércoles
3 → jueves
4 → viernes
5 → sábado
6 → domingo
```

El nombre del día resulta más fácil de leer, mientras que el número puede ser útil para ordenar los días correctamente o para construir algunos tipos de análisis.

En este sección trabajaremos con usar estas columnas solo para primeras exploraciones temporales. Más adelante podremos profundizar en agrupamientos, resúmenes y visualizaciones por períodos.

### Punto de control

El rango temporal permite detectar periodos inesperados, fechas fuera de contexto o concentraciones que pueden influir en la lectura de las ventas.

## Explorar la distribución temporal

Después de crear columnas como `anio`, `mes`, `dia` y `dia_semana`, es posible empezar a hacer preguntas simples sobre la distribución temporal de las transacciones.

Por ejemplo, es posible revisar cuántas transacciones hay por año.

### Punto de control

Las columnas derivadas no sustituyen a la fecha original. Funcionan como variables auxiliares para responder preguntas específicas sobre años, meses, días y patrones de actividad.

In [ ]:
df_temporal["anio"].value_counts(dropna=False)

Este conteo nos permite ver si el dataset contiene registros de un solo año o de varios.

También es posible contar transacciones por mes:

### Punto de control

Cuando se agrupan transacciones, el nivel de detalle cambia: se pasa de registros individuales a resúmenes por fecha o periodo. Esa transformación debe interpretarse con claridad.

In [ ]:
df_temporal["mes"].value_counts(dropna=False).sort_index()

Para este ejemplo usamos `sort_index()` para que los meses aparezcan ordenados numéricamente, del 1 al 12.

Este tipo de conteo todavía es simple, pero ya muestra el valor de haber convertido la columna de fecha. A continuación es posible analizar partes de la fecha que antes no estaban disponibles como columnas.

También es posible revisar las transacciones por día de la semana:

### Punto de control

Una validación temporal debe revisar tanto el tipo de columna como los valores extremos, los nulos y la coherencia del orden cronológico.

In [ ]:
df_temporal["dia_semana"].value_counts(dropna=False)

Este conteo muestra cuántas transacciones corresponden a cada día de la semana.

Es posible que los días no aparezcan en orden calendario, porque `value_counts()` ordena por frecuencia de manera descendente. Más adelante podremos trabajar formas más ordenadas de presentar este tipo de resultados.

Por ahora, lo relevante es notar que la columna de fecha nos permitió construir nuevas variables y empezar a explorar patrones temporales.

### Punto de control

Una fecha almacenada como texto puede parecer correcta en una muestra y aun así impedir operaciones temporales. El tipo de dato es el que habilita ordenar, comparar y agrupar cronológicamente.

## Resumir transacciones por fecha

Hasta ahora usamos `value_counts()` para contar transacciones según año, mes o día de la semana.

Otra forma de resumir información por grupos es usar `groupby()`.

`groupby()` es una herramienta muy potente de Pandas. Permite agrupar filas según los valores de una o más columnas y luego calcular resúmenes para cada grupo. Más adelante trabajaremos con estudiarla con mayor profundidad.

En este sección la trabajaremos con usar solo de manera introductoria, para mostrar cómo una columna temporal puede servir para resumir información por períodos.

Por ejemplo, es posible contar cuántas transacciones hubo en cada fecha.

### Punto de control

La conversión se realiza sobre una copia para conservar una referencia del estado original. Esta separación facilita revisar qué valores cambiaron y qué casos requieren atención adicional.

In [ ]:
transacciones_por_fecha = (
    df_temporal
    .groupby("Transaction Date")["Transaction ID"]
    .count()
)

transacciones_por_fecha.head()

En esta instrucción agrupamos el dataset por `Transaction Date` y contamos cuántos identificadores de transacción aparecen en cada fecha.

El resultado muestra la cantidad de transacciones registradas por día.

También es posible ordenar ese resultado para ver los días con mayor cantidad de transacciones:

### Punto de control

Usar `errors="coerce"` evita que una sola cadena defectuosa detenga todo el proceso, pero convierte esos casos en `NaT`. Por eso siempre debe acompañarse de un conteo y una revisión.

In [ ]:
transacciones_por_fecha.sort_values(ascending=False).head(10)

Este resultado permite identificar las fechas con más transacciones registradas.

El uso de `groupby()` nos permite pasar de una tabla de transacciones individuales a un resumen por período. Esa es una de las razones por las que convertir correctamente las fechas es tan relevante: una vez que Pandas entiende la columna como temporal, es posible construir análisis organizados por día, mes, año u otros períodos.

### Punto de control

Las fechas no interpretables no deben desaparecer automáticamente. Pueden conservar información útil en otras columnas y deberán tratarse según el objetivo del análisis.

## Revisar registros sin fecha válida

Cuando convertimos `Transaction Date`, vimos que no todas las filas quedaron con una fecha válida.

Antes de la conversión había valores faltantes reales. Después de convertir con `errors="coerce"`, algunos valores adicionales pasaron a ser `NaT` porque no pudieron interpretarse como fechas.

Esto es relevante porque las filas con `NaT` no pueden ubicarse correctamente en una línea de tiempo. No pueden asignarse a un año, a un mes, a un día ni a un día de la semana.

Podemos revisar cuántas fechas no interpretables o faltantes quedaron en `df_temporal`.

### Punto de control

El rango temporal permite detectar periodos inesperados, fechas fuera de contexto o concentraciones que pueden influir en la lectura de las ventas.

In [ ]:
df_temporal["Transaction Date"].isna().sum()

Ese conteo nos indica cuántas filas no tienen una fecha válida después de la conversión.

También es posible observar algunas de esas filas.

### Punto de control

Las columnas derivadas no sustituyen a la fecha original. Funcionan como variables auxiliares para responder preguntas específicas sobre años, meses, días y patrones de actividad.

In [ ]:
df_temporal[df_temporal["Transaction Date"].isna()].head(10)

Estas filas no deberían desaparecer automáticamente del dataset. Pueden seguir siendo útiles para otros análisis, como productos vendidos, métodos de pago o importes, siempre que esas columnas tengan datos válidos.

Aun así, para análisis temporales específicos, estas filas requieren una decisión. Podemos excluirlas de ciertos conteos por fecha, conservarlas como registros sin fecha, revisar si el valor original era `"UNKNOWN"` o `"ERROR"`, o intentar recuperar la fecha desde otra fuente si existiera.

La decisión depende del contexto y del objetivo del análisis.

En este sección no trabajaremos con corregir esas fechas. Lo relevante es reconocer que una conversión temporal puede dejar filas fuera del análisis por período, y que eso debe ser registrado.

### Punto de control

Cuando se agrupan transacciones, el nivel de detalle cambia: se pasa de registros individuales a resúmenes por fecha o periodo. Esa transformación debe interpretarse con claridad.

## Precauciones al analizar fechas

Al trabajar con fechas, uno de los errores más frecuentes es asumir que una columna es temporal solo porque sus valores se ven como fechas.

Una fecha escrita como `"2023-09-08"` puede parecer correcta para una persona, pero si Pandas la cargó como `object`, todavía es texto. En ese estado, la columna no permite aprovechar correctamente las herramientas temporales de Pandas.

Por esa razón, antes de analizar fechas, resulta útil revisar el tipo de dato:

```python
df["Transaction Date"].dtype
```

Otro error frecuente es convertir fechas sin verificar cuántos valores quedaron como `NaT`. Cuando usamos `pd.to_datetime()` con `errors="coerce"`, los valores que no pueden interpretarse como fechas se convierten en `NaT`. Esto evita que el código se detenga, pero también puede ocultar problemas si no revisamos el resultado.

Por esa razón, después de convertir, resulta útil comparar faltantes antes y después:

```python
df["Transaction Date"].isna().sum()
df_fechas["Transaction Date"].isna().sum()
```

También puede ser un error eliminar automáticamente las filas con fechas faltantes o no interpretables. Esas filas no sirven para ciertos análisis temporales, pero pueden seguir siendo útiles para otros análisis. Por ejemplo, una venta sin fecha válida todavía podría aportar información sobre producto, cantidad, importe o método de pago.

Otro punto relevante es no abrir demasiadas conclusiones solo a partir de conteos temporales simples. Saber que un día tuvo más transacciones que otro puede ser una señal interesante, pero antes de sacar conclusiones deberíamos considerar el contexto: cantidad de días registrados, posibles datos faltantes, períodos incompletos o sesgos de carga.

Finalmente, cuando extraemos columnas como `anio`, `mes`, `dia` o `dia_semana`, debemos recordar que esas columnas dependen de que la fecha original esté correctamente convertida. Si una fila tiene `NaT`, las columnas derivadas también quedarán sin información válida.

Una buena rutina para trabajar con fechas podría ser:

```text
revisar el tipo de dato original
convertir la columna con pd.to_datetime()
comparar faltantes antes y después
revisar valores no interpretables
analizar el rango temporal
crear columnas temporales derivadas
verificar los resultados antes de sacar conclusiones
```

Trabajar con fechas no consiste solamente en cambiar el tipo de una columna. Implica asegurarse de que los valores temporales sean interpretables, revisar qué registros quedan fuera del análisis temporal y usar las nuevas variables con cuidado.

### Punto de control

Una validación temporal debe revisar tanto el tipo de columna como los valores extremos, los nulos y la coherencia del orden cronológico.

## Síntesis del análisis temporal

En esta práctica trabajamos con fechas y datos temporales.

Partimos de una idea central: una fecha escrita como texto no es lo mismo que una fecha interpretada como dato temporal por Pandas. Aunque valores como `"2023-09-08"` parezcan fechas válidas a simple vista, si la columna está cargada como `object`, Pandas todavía la trata como texto.

Primero revisamos la columna `Transaction Date` y comprobamos que había sido cargada como `object`:

```python
df["Transaction Date"].dtype
```

También obsertrabajaremos conlgunos valores y contamos los faltantes antes de la conversión. En ese momento detectamos 159 valores faltantes reales en la columna original.

Luego convertimos la columna usando `pd.to_datetime()`:

```python
df_fechas["Transaction Date"] = pd.to_datetime(
    df_fechas["Transaction Date"],
    errors="coerce"
)
```

El parámetro `errors="coerce"` permitió convertir en `NaT` los valores que no podían interpretarse como fechas. Después de la conversión, la cantidad de valores faltantes temporales aumentó a 460. Esto indicó que, además de los faltantes originales, había valores escritos que no podían interpretarse como fechas.

Para investigar esos casos, construimos una condición que identificó las filas donde la columna original no estaba vacía, pero la columna convertida quedó como `NaT`:

```python
fechas_no_convertidas = (
    df["Transaction Date"].notna()
    & df_fechas["Transaction Date"].isna()
)
```

Ese paso nos permitió diferenciar entre valores que ya estaban faltantes antes de convertir y valores que se volvieron `NaT` durante la conversión.

Después revisamos el rango temporal del dataset usando la fecha mínima y la fecha máxima. Esto nos permitió saber qué período cubren los datos y también pensar en posibles valores fuera de rango.

Más adelante ordenamos el dataset por fecha y vimos que, una vez convertida la columna, Pandas puede ordenar cronológicamente los registros.

Luego creamos nuevas columnas temporales a partir de `Transaction Date`:

```python
df_temporal["anio"] = df_temporal["Transaction Date"].dt.year
df_temporal["mes"] = df_temporal["Transaction Date"].dt.month
df_temporal["dia"] = df_temporal["Transaction Date"].dt.day
```

También extraímos el día de la semana:

```python
df_temporal["dia_semana"] = df_temporal["Transaction Date"].dt.day_name()
df_temporal["numero_dia_semana"] = df_temporal["Transaction Date"].dt.dayofweek
```

Estas nuevas columnas permiten hacer preguntas temporales más específicas, como cuántas transacciones hubo por año, por mes o por día de la semana.

Finalmente hicimos una primera agrupación temporal con `groupby()`. Usamos esta herramienta de manera introductoria para contar transacciones por fecha:

```python
transacciones_por_fecha = (
    df_temporal
    .groupby("Transaction Date")["Transaction ID"]
    .count()
)
```

Más adelante trabajaremos con estudiar `groupby()` con mayor profundidad. En este sección lo usamos solo para mostrar que una columna temporal bien convertida permite resumir información por períodos.

La idea principal de este sección fue:

```text
Trabajar con fechas requiere convertirlas, verificarlas y luego usarlas para construir nuevas variables temporales.
```

Las fechas permiten analizar evolución, períodos, patrones y distribución temporal. Pero antes de hacerlo, es necesario asegurarnos de que Pandas las interprete correctamente y de que sepamos qué registros quedaron sin fecha válida.

### Punto de control

Una fecha almacenada como texto puede parecer correcta en una muestra y aun así impedir operaciones temporales. El tipo de dato es el que habilita ordenar, comparar y agrupar cronológicamente.